# 使用 FAISS 进行语义搜索 (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install faiss-gpu

In [ ]:
from huggingface_hub import hf_hub_url

data_files = hf_hub_url(
    repo_id="lewtun/github-issues",
    filename="datasets-issues-with-comments.jsonl",
    repo_type="dataset",
)

在本节，我们将使用这些信息**构建一个搜索引擎**，帮助我们找到关于该库的最相关的 issue 的答案！

##使用文本嵌入进行语义搜索

基于 Transformer 的语言模型会将文本中的每个 token 转换为嵌入向量。事实证明，我们可以 “池化（pool）” 嵌入向量以创建整个句子、段落或（在某些情况下）文档的向量表示。然后，通过计算每个嵌入之间的点积相似度（或其他一些相似度度量）并返回相似度最大的文档，这些嵌入可用于在语料库中找到相似的文档。

###加载和准备数据集

In [ ]:
from datasets import load_dataset

issues_dataset = load_dataset("json", data_files=data_files, split="train")
issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 2855
})

In [ ]:
issues_dataset = issues_dataset.filter(
    lambda x: (x["is_pull_request"] == False and len(x["comments"]) > 0)
)
issues_dataset

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 771
})

我们可以看到，我们的数据集中有很多列，其中大部分在构建我们的搜索引擎都不会使用。从搜索的角度来看，信息量最大的列是 title ， body ，和 comments ，而 html_url 为我们提供了一个回到原 issue 的链接。让我们使用 `Dataset.remove_columns()` 删除其余的列：

In [ ]:
columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns_to_keep).symmetric_difference(columns)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 771
})

为了创建我们的文本嵌入数据集，我们将**用 issue 的标题和正文来扩充每条评论**，因为这些字段通常包含有用的上下文信息。因为我们的 comments 列当前是每个 issue 的评论列表，我们**需要“重新组合”列，使得每一行都是由一个 (html_url, title, body, comment) 元组组成**。在 Pandas 中，我们可以使用 `DataFrame.explode()` 函数 完成这个操作。

它**为类似列表的列中的每个元素创建一个新行，同时复制所有其他列值。**让我们首先切换到 Pandas 的 DataFrame 格式：

In [ ]:
issues_dataset.set_format("pandas")
df = issues_dataset[:]

In [ ]:
df["comments"][0].tolist()

['the bug code locate in ：\r\n    if data_args.task_name is not None:\r\n        # Downloading and loading a dataset from the hub.\r\n        datasets = load_dataset("glue", data_args.task_name, cache_dir=model_args.cache_dir)',
 'Hi @jinec,\r\n\r\nFrom time to time we get this kind of `ConnectionError` coming from the github.com website: https://raw.githubusercontent.com\r\n\r\nNormally, it should work if you wait a little and then retry.\r\n\r\nCould you please confirm if the problem persists?',
 'cannot connect，even by Web browser，please check that  there is some  problems。',
 'I can access https://raw.githubusercontent.com/huggingface/datasets/1.7.0/datasets/glue/glue.py without problem...']

我们**希望使用 `explode()` 将这些评论中的每一条都展开成为一行**。

1. `df.explode("comments", ignore_index=True)`
   - `explode("comments")`：把 `df` 中 `comments` 列里**列表/数组类型**的数据拆分成多行，列表里每个元素单独占一行，其他列数据同步复制；
   - `ignore_index=True`：拆分后重置行索引，从0开始连续编号，不保留原错乱索引。
2. 赋值给 `comments_df`：生成拆分后的新数据表。
3. `comments_df.head(4)`：打印新表前4行数据，用于预览拆分效果。

### 场景举例
原表一行 `comments=[评论1,评论2]`，执行后会变成两行，分别对应两条独立评论，方便单条评论分析。

原始数据一行对应一个 issue，comments是评论列表（一条 issue 多条评论挤在同一个单元格）；

explode("comments")会把列表里每条评论拆成独立新行；
拆分时会复制该行剩余字段（html_url、title、body），保证每条拆分后的评论都绑定所属 issue 标题、正文、链接；

最终效果：每行仅存单条评论，配套完整上下文信息，方便后续单独对每条评论做向量化检索。

In [ ]:
comments_df = df.explode("comments", ignore_index=True)
comments_df.head(4)

创建一个新的 comments_length 列来存放每条评论的字数：

In [ ]:
from datasets import Dataset

comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 2842
})

使用这个新列来过滤掉简短的评论

In [ ]:
comments_dataset = comments_dataset.map(
    lambda x: {"comment_length": len(x["comments"].split())}
)

In [ ]:
comments_dataset = comments_dataset.filter(lambda x: x["comment_length"] > 15)
comments_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length'],
    num_rows: 2098
})

目的：**把 issue 完整上下文整合到单一文本字段**，用于后续生成统一语义嵌入向量，让搜索引擎同时匹配标题、问题描述、评论里的相关信息，避免只匹配单一片段丢失上下文。

拼接逻辑：将三部分用**换行符拼接**：标题 + \n + issue正文(body) + \n + 单条评论。

前置基础：数据集已拆分成一行一条评论，**每条记录自带对应 issue 的标题、完整问题描述，三者合并构成完整问答上下文**。

作用：后续模型只需对这一列text编码，就能得到包含问题 + 回复全部信息的语义向量，**提升语义检索准确度**。

In [ ]:
def concatenate_text(examples):
    return {
        "text": examples["title"]
        + " \n "
        + examples["body"]
        + " \n "
        + examples["comments"]
    }


comments_dataset = comments_dataset.map(concatenate_text)

##创建文本嵌入

我们可以通过使用 `AutoModel` 类来**完成文本嵌入**。（加载完整 Transformer 编码器，输出每一字 token 的隐藏层向量，作为文本嵌入的原始素材）首先需要做的就是选择一个合适的 checkpoint。幸运的是，有一个名为 **sentence-transformers** 的库专门用于创建文本嵌入。

我们这次要实现的是**非对称语义搜索**（检索两端文本长度、表达形式不对等的语义匹配，和 “查询 - 文档等长 / 同类型” 的对称语义搜索相对。），因为我们有一个简短的查询，我们希望在比如 issue 评论等更长的文档中找到其匹配的文本。通过查看 模型概述表 我们可以发现 multi-qa-mpnet-base-dot-v1 checkpoint 在语义搜索方面具有最佳性能，因此我们将使用它。我们还将使用这个 checkpoint 加载了对应的 tokenizer ：

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)

将模型和输入的文本放到 GPU 上会加速嵌入过程

In [ ]:
import torch

device = torch.device("cuda")
model.to(device)

根据我们之前的想法，我们希望将我们的 GitHub issue 中的每一条记录转化为一个单一的向量，**所以我们需要以某种方式“池化（pool）”或平均每个词的嵌入向量。**

一种流行的方法是在我们模型的输出上进行 CLS 池化 。Transformer 输入开头会添加特殊[CLS]标记，训练时该 token 会学习存储整句全局语义；直接取它最后一层隐藏状态，就能得到整段文本的统一向量，不用对所有 token 做平均。**我们只需要收集 [CLS] token 的的最后一个隐藏状态。** 以下函数实现了这个功能：

In [ ]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

In [ ]:
def get_embeddings(text_list):
    encoded_input = tokenizer(
        text_list, padding=True, truncation=True, return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

In [ ]:
embedding = get_embeddings(comments_dataset["text"][0])
embedding.shape

torch.Size([1, 768])

太好了，我们已经将语料库中的第一个条目转换为了一个 768 维向量！我们可以用 Dataset.map() 将我们的 `get_embeddings()` 函数应用到我们语料库中的每一行，然后创建一个新的 embeddings 列。

In [ ]:
embeddings_dataset = comments_dataset.map(
    lambda x: {"embeddings": get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

我们已经将文本嵌入转换为 NumPy 数组——这是因为当我们尝试使用 FAISS 搜索它们时，🤗 Datasets 需要这种格式。

##使用 FAISS 进行高效的相似性搜索

 FAISS （Facebook AI Similarity Search 的缩写）是一个库，提供了用于快速搜索和聚类嵌入向量的高效算法。

FAISS 背后的基本思想是创建一个特殊的数据结构，称为 index（索引） 它可以**找到哪些嵌入与输入嵌入相似**。在 🤗 Datasets 中创建一个 FAISS index（索引）很简单——我们使用 `Dataset.add_faiss_index()` 函数并指定我们要索引的数据集的哪一列：

In [ ]:
embeddings_dataset.add_faiss_index(column="embeddings")

我们可以使用 `Dataset.get_nearest_examples()` 函数进行最近邻居查找。

In [ ]:
question = "How can I load a dataset offline?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

torch.Size([1, 768])

In [ ]:
scores, samples = embeddings_dataset.get_nearest_examples(
    "embeddings", question_embedding, k=5
)

Dataset.get_nearest_examples() 函数返回一个元组，包括评分（评价查询和文档之间的相似程度）和对应的样本（这里是 5 个最佳匹配）。让我们把这些收集到一个 pandas.DataFrame ，这样我们就可以轻松地对它们进行排序：

In [ ]:
import pandas as pd

samples_df = pd.DataFrame.from_dict(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)

In [ ]:
for _, row in samples_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.scores}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()

"""
COMMENT: Requiring online connection is a deal breaker in some cases unfortunately so it'd be great if offline mode is added similar to how `transformers` loads models offline fine.

@mandubian's second bullet point suggests that there's a workaround allowing you to use your offline (custom?) dataset with `datasets`. Could you please elaborate on how that should look like?
SCORE: 25.505046844482422
TITLE: Discussion using datasets in offline mode
URL: https://github.com/huggingface/datasets/issues/824

COMMENT: The local dataset builders (csv, text , json and pandas) are now part of the `datasets` package since #1726 :)
You can now use them offline
\`\`\`python
datasets = load_dataset("text", data_files=data_files)
\`\`\`

We'll do a new release soon
SCORE: 24.555509567260742
TITLE: Discussion using datasets in offline mode
URL: https://github.com/huggingface/datasets/issues/824

COMMENT: I opened a PR that allows to reload modules that have already been loaded once even if there's